# Final Dataset Construction

This notebook combines all engineered features produced during the data preparation phase.

Sources:
- listings_features.csv
- review_features.csv

The resulting dataset will be used for machine learning models such as Decision Trees and Random Forests.

In [225]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: /Users/santiagotawata/Desktop/airbnb-rome-analysis


## 1. Load Engineered Datasets

The datasets generated in Notebook 02 and Notebook 03 are loaded and prepared for integration.

- listings_features.csv
- review_features.csv

In [226]:
listings_features = pd.read_csv(
    "../data/listings_features.csv"
)

review_features = pd.read_csv(
    "../data/review_features.csv"
)

## 2. Dataset Inspection

Before merging the datasets, we verify their dimensions, structure, and merge keys.

In [227]:
print(listings_features.shape)
print(review_features.shape)

(33564, 68)
(32826, 9)


In [228]:
listings_features.head()

,id,price,latitude,longitude,accommodates,bedrooms,beds,bathrooms,host_is_superhost,host_response_rate,...,neighbourhood_cleansed_VII San Giovanni/Cinecittà,neighbourhood_cleansed_VIII Appia Antica,neighbourhood_cleansed_X Ostia/Acilia,neighbourhood_cleansed_XI Arvalia/Portuense,neighbourhood_cleansed_XII Monte Verde,neighbourhood_cleansed_XIII Aurelia,neighbourhood_cleansed_XIV Monte Mario,neighbourhood_cleansed_XV Cassia/Flaminia,distance_to_colosseum,location_cluster
0,2737,57.0,41.871360,12.482150,1,1.0,1.0,1.5,0,0.0,...,0,1,0,0,0,0,0,0,2.252726,3.0
1,11834,110.0,41.895447,12.491181,2,1.0,1.0,1.0,1,100.0,...,0,0,0,0,0,0,0,0,0.588865,4.0
2,12398,124.0,41.925820,12.469280,6,2.0,3.0,1.0,1,100.0,...,0,0,0,0,0,0,0,0,4.389669,1.0
3,19965,162.0,41.908230,12.452930,5,2.0,3.0,1.0,0,100.0,...,0,0,0,0,0,0,0,0,3.824847,1.0
4,19967,150.0,41.908283,12.452617,5,2.0,3.0,1.0,0,100.0,...,0,0,0,0,0,0,0,0,3.850102,1.0


In [229]:
review_features.head()

,listing_id,review_count,avg_review_length,avg_sentiment_score,sentiment_label,latest_review_year,latest_review_month,latest_review_day,days_since_latest_review
0,2737,5,51.800000,0.956560,positive,2015,5,8,4077
1,11834,302,74.165563,0.849785,positive,2026,6,7,29
2,12398,85,84.658824,0.719140,positive,2025,8,1,339
3,19965,195,44.953846,0.463973,positive,2026,6,13,23
4,20534,50,60.200000,0.730592,positive,2022,11,22,1322


### 2.1 Verify Merge Keys

The listings dataset uses the variable id as its primary identifier, while the review dataset uses listing_id.

We verify that both identifiers exist before performing the merge operation.

In [230]:
print("id" in listings_features.columns)
print("listing_id" in review_features.columns)

True
True


## 3. Merge Listings and Review Features

Review-derived variables are merged with the listings dataset using the Airbnb listing identifier.

A left join is used so that all listings remain in the final dataset, even if some properties have never received reviews.

In [231]:
final_df = listings_features.merge(
    review_features,
    left_on="id",
    right_on="listing_id",
    how="left"
)

#deletion of duplicate key
final_df.drop(
    columns=["listing_id"],
    inplace=True
)

In [232]:
print(final_df.shape)

(33564, 76)


In [233]:
final_df.columns.to_list()

['id',
 'price',
 'latitude',
 'longitude',
 'accommodates',
 'bedrooms',
 'beds',
 'bathrooms',
 'host_is_superhost',
 'host_response_rate',
 'host_acceptance_rate',
 'review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_location',
 'review_scores_value',
 'number_of_reviews',
 'reviews_per_month',
 'minimum_nights',
 'maximum_nights',
 'beds_per_guest',
 'bathrooms_per_guest',
 'amenities_count',
 'has_wifi',
 'has_air_conditioning',
 'has_kitchen',
 'has_washer',
 'has_dryer',
 'has_parking',
 'has_elevator',
 'has_tv',
 'has_workspace',
 'host_experience_days',
 'professional_host',
 'property_type_Dome',
 'property_type_Entire Home',
 'property_type_Hotel',
 'property_type_Private Room',
 'property_type_Private room in boat',
 'property_type_Shared Room',
 'property_type_Shared room in loft',
 'room_type_Hotel room',
 'room_type_Private room',
 'room_type_Shared room',
 'instant_bookable_t',
 'review_quality_index',
 'listing_age_days',
 'review_recency_days',
 're

## 3. Data Preprocessing


### 3.1 Missing Value Treatment

Remaining missing numerical values are replaced using median imputation.

In [234]:
final_df.isnull().sum().sort_values(
    ascending=False
).head(20)

days_since_latest_review                   6887
latest_review_day                          6887
latest_review_month                        6887
latest_review_year                         6887
sentiment_label                            6887
avg_sentiment_score                        6887
avg_review_length                          6887
review_count                               6887
review_recency_days                           0
neighbourhood_cleansed_I Centro Storico       0
demand_proxy                                  0
occupancy_rate                                0
days_since_last_review                        0
review_intensity                              0
room_type_Shared room                         0
listing_age_days                              0
review_quality_index                          0
instant_bookable_t                            0
neighbourhood_cleansed_III Monte Sacro        0
room_type_Private room                        0
dtype: int64

Columns where NaN means "no reviews", not "missing data

In [ ]:

review_related_cols = [
    "review_count",
    "avg_review_length",
    "avg_sentiment_score",
    "sentiment_label",
    "latest_review_year",
    "latest_review_month",
    "latest_review_day",
    "days_since_latest_review",
]






Median imputation, excluding review-related columns

In [ ]:
numeric_cols = final_df.select_dtypes(include=np.number).columns
numeric_cols = [c for c in numeric_cols if c not in review_related_cols]

for col in numeric_cols:
    final_df[col] = final_df[col].fillna(final_df[col].median())



 Specific treatment for listings without reviews

In [ ]:

final_df["review_count"] = final_df["review_count"].fillna(0)
final_df["avg_review_length"] = final_df["avg_review_length"].fillna(0)
final_df["avg_sentiment_score"] = final_df["avg_sentiment_score"].fillna(0)
final_df["sentiment_label"] = final_df["sentiment_label"].fillna("no_reviews")



Days_since_last_Review (9999 = no reviews) 

In [ ]:

final_df["days_since_latest_review"] = final_df["days_since_latest_review"].fillna(9999)

Date components (if year/month/day already exist) 

In [ ]:

for col in ["latest_review_year", "latest_review_month", "latest_review_day"]:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(-1)

print("Remaining null values in review-related columns:")
print(final_df[review_related_cols].isnull().sum())

Remaining null values in review-related columns:
review_count                0
avg_review_length           0
avg_sentiment_score         0
sentiment_label             0
latest_review_year          0
latest_review_month         0
latest_review_day           0
days_since_latest_review    0
dtype: int64


### 3.2 Date Feature Engineering

Date variables cannot be directly used by machine learning algorithms.

We therefore transform review dates into numerical variables that capture review recency and temporal patterns.

#### Latest Review Date Components

In [239]:
# df["latest_review_date"] = pd.to_datetime(
#     df["latest_review_date"],
#     errors="coerce"
# )

# df["latest_review_year"] = (
#     df["latest_review_date"].dt.year
# )

# df["latest_review_month"] = (
#     df["latest_review_date"].dt.month
# )

# df["latest_review_day"] = (
#     df["latest_review_date"].dt.day
# )

# df.drop(
#     columns=["latest_review_date"],
#     inplace=True
# )

#### Review Recency

In [240]:
# df["days_since_last_review"] = (
#     pd.Timestamp.today() - pd.to_datetime(df["last_review"])
# ).dt.days

In [241]:
# df.drop(
#     columns=["last_review"],
#     inplace=True
# )

### 3.3 Identifier Removal
Unique identifiers do not provide predictive information for price estimation and may introduce unnecessary noise into the model.
Therefore, the listing identifier is removed before training.

In [242]:
# adjust to dataset
columns_to_drop = [
    "id"
]

existing_cols = [c for c in columns_to_drop if c in final_df.columns]

final_df = final_df.drop(columns=existing_cols)

## 4. Handle Missing Review Features
Listings without reviews do not appear in the reviews dataset.

For these cases:

- review_count is set to 0
- avg_review_length is set to 0

This indicates the absence of review activity rather than missing information.

In [243]:
# final_df.isna().sum()

In [244]:
# final_df["review_count"] = (
#     final_df["review_count"]
#     .fillna(0)
# )

# final_df["avg_review_length"] = (
#     final_df["avg_review_length"]
#     .fillna(0)
# )

## 5. Additional Review Recency Feature

The latest review date is transformed into a numerical variable representing the number of days since the most recent review.

Listings without reviews receive a large placeholder value (9999) to indicate very low review recency.

In [245]:
# final_df["latest_review_date"] = pd.to_datetime(
#     final_df["latest_review_date"]
# )

In [246]:
# reference_date = final_df["latest_review_date"].max()

# final_df["days_since_latest_review"] = (
#     reference_date - final_df["latest_review_date"]
# ).dt.days

In [247]:
# final_df["days_since_latest_review"] = (
#     final_df["days_since_latest_review"]
#     .fillna(9999)
# )

## 6. Final Dataset Validation

Before exporting the final dataset, we verify:

- dataset dimensions
- variable types
- remaining missing values

In [248]:
print(final_df.shape)

(33564, 75)


In [249]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33564 entries, 0 to 33563
Data columns (total 75 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   price                                              33564 non-null  float64
 1   latitude                                           33564 non-null  float64
 2   longitude                                          33564 non-null  float64
 3   accommodates                                       33564 non-null  int64  
 4   bedrooms                                           33564 non-null  float64
 5   beds                                               33564 non-null  float64
 6   bathrooms                                          33564 non-null  float64
 7   host_is_superhost                                  33564 non-null  int64  
 8   host_response_rate                                 33564 non-null  float64
 9   host_a

## 7. Export Final Dataset

The final integrated dataset is exported and will serve as the input for the machine learning models developed in Notebook 06.

In [251]:
final_df.to_csv(
    "../data/final_dataset.csv",
    index=False
)

print("final_dataset.csv saved")
print(final_df.shape)

final_dataset.csv saved
(33564, 75)
